In [10]:
import pandas as pd
import numpy as np

In [11]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Bawana, Delhi - DPCC.xlsx",skiprows=16)

In [12]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [13]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [14]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

In [15]:
# ---------- 5. Convert date columns to datetime ----------
if 'From Date' in df.columns:
    df['From Date'] = pd.to_datetime(df['From Date'], errors='coerce')
if 'To Date' in df.columns:
    df['To Date'] = pd.to_datetime(df['To Date'], errors='coerce')

In [16]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 20)
   From Date    To Date   PM2.5    PM10     NO    NO2    NOx     NH3   SO2  \
0 2025-01-01 2025-02-01  187.26  250.42   6.16  22.15  16.80  34.240  7.49   
1 2025-02-01 2025-03-01  200.71  275.88   7.33  20.31  16.57  26.745  6.82   
2 2025-03-01 2025-04-01  240.30  317.74   4.63  25.46  27.56  26.745  8.82   
3 2025-04-01 2025-05-01  246.25  313.33   4.63  36.70  31.50  26.745  7.62   
4 2025-05-01 2025-06-01  212.12  307.08  10.24  35.82  27.38  26.745  8.87   

      CO  Ozone  Benzene  Toluene     RH    WS      WD      BP     AT    RF  \
0  0.970  20.81     1.87    12.88  82.14  0.78  169.61  998.91  13.13  0.00   
1  1.430  20.09     1.84     9.06  83.31  0.73  174.31  998.96  13.27  0.01   
2  0.885  16.61     2.73    24.28  84.83  0.70  178.99  998.75  14.26  0.00   
3  0.885  13.15     3.89    25.78  85.61  1.02  238.34  998.85  14.34  0.01   
4  1.660  19.30     2.43     9.69  77.80  1.22  271.30  998.66  14.78  0.01   

   TOT-RF  
0     0.0  
1     1.0

In [17]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [20]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,2025-01-01,2025-02-01,1.709958,0.676004,0.969144,0.190213,0.359523,0.928380,-0.340262,0.129095,-0.805528,0.381498,-0.028122,1.216503,-1.202675,-0.212842,2.211584,-2.195142,-0.434472,-0.439168
1,2025-02-01,2025-03-01,1.947376,0.927672,1.675445,0.028037,0.326179,-0.120837,-0.468278,1.599334,-0.843248,0.354759,-0.434747,1.307452,-1.313063,-0.097803,2.224635,-2.173523,0.587814,0.748775
2,2025-03-01,2025-04-01,2.646213,1.341450,0.045521,0.481954,1.919460,-0.120837,-0.086139,-0.142579,-1.025561,1.148003,1.185368,1.425608,-1.379296,0.016746,2.169822,-2.020650,-0.434472,-0.439168
3,2025-04-01,2025-05-01,2.751241,1.297858,0.045521,1.472639,2.490663,-0.120837,-0.315423,-0.142579,-1.206826,2.181894,1.345037,1.486240,-0.672814,1.469418,2.195923,-2.008297,0.587814,0.629981
4,2025-05-01,2025-06-01,2.148783,1.236078,3.432141,1.395077,1.893364,-0.120837,-0.076586,2.334453,-0.884635,0.880618,-0.367686,0.879138,-0.231262,2.276158,2.146330,-1.940353,0.587814,0.154804
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,2025-12-11,NaT,-0.222566,-0.120415,0.280954,1.848113,1.657054,0.089847,1.566611,3.005649,1.102995,-0.946517,-1.188388,-0.412024,-0.871512,1.471376,1.144027,-1.264004,-0.434472,-0.439168
316,NaT,NaT,-0.222566,3.017816,0.878593,2.005001,1.928158,-0.225128,1.893339,2.462300,0.640402,-0.937604,-1.249062,-0.054448,-1.489683,1.426339,1.209281,-1.333492,-0.434472,-0.439168
317,NaT,NaT,-0.222566,2.312732,-0.358942,1.435621,1.196032,-0.512106,3.020649,1.631296,0.884010,-0.964343,-1.282061,-0.226240,-1.202675,1.383506,1.230162,-1.342757,-0.434472,-0.439168
318,NaT,NaT,-0.222566,2.332502,0.250771,1.633934,1.494682,-0.493907,0.844368,2.174645,0.819048,-0.955430,-1.263965,-0.303197,-1.070210,1.516412,1.188400,-1.348934,-0.434472,-0.439168


In [19]:
df.to_excel('Bawana2025.xlsx', index=False)